# T4 DSA+SEA+Snapshot Cross-Dataset Runner

This notebook assumes the preprocessed Drive files already exist and only loads them for training.

Required legacy sfreq100 files:
- `/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100/cho2017.npz`
- `/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100/lee2019.npz`

Expected shapes from the existing cross-dataset matrix: Cho2017 `(10520, 64, 201)`, Lee2019 `(10800, 62, 201)`. Do not use a Lee2019 `(5400, 62, 201)` one-session export.

Runtime: select `T4 GPU` before running.

## 1. Runtime And Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi

Mounted at /content/drive
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Sun Jun 28 11:49:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|       

## 2. Clone Or Update Repository

In [2]:
import os

REPO_URL = 'https://github.com/heegyukim4043/MI_loso_project.git'
REPO_DIR = '/content/MI_loso_project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}/project/crossdata/models
!git rev-parse --short HEAD

Cloning into '/content/MI_loso_project'...
remote: Enumerating objects: 511, done.
remote: Counting objects: 100% (511/511), done.
remote: Compressing objects: 100% (337/337), done.
remote: Total 511 (delta 178), reused 475 (delta 154), pack-reused 0 (from 0)
Receiving objects: 100% (511/511), 758.79 KiB | 11.67 MiB/s, done.
Resolving deltas: 100% (178/178), done.
/content/MI_loso_project/project/crossdata/models
09da921


## 3. Set Data Path And Verify Files

No MOABB download or preprocessing is run here. The notebook stops if the two `.npz` files are missing or malformed.

In [4]:
import os
import numpy as np

os.environ['MI_PREPROCESSED_DIR'] = '/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100'
os.environ['MI_BACKUP_DIR'] = '/content/drive/MyDrive/MI_loso_project/colab_results/dsa_sea_snapshot_20260628'
os.environ['MI_N_TIMES'] = '201'

prep = os.environ['MI_PREPROCESSED_DIR']
print('MI_PREPROCESSED_DIR =', prep)
print('MI_BACKUP_DIR       =', os.environ['MI_BACKUP_DIR'])
!ls -lh "$MI_PREPROCESSED_DIR"

# ── 검증: 실제 데이터 확인 후 기존 프로토콜과 일치하는지 체크 ──────────────────
REQUIRED = {
    'cho2017': {'n_subjects': 52, 'n_ch': 64, 'n_times': 201,
                'sfreq': 100.0, 'trials_per_subject': None},
    'lee2019': {'n_subjects': 54, 'n_ch': 62, 'n_times': 201,
                'sfreq': 100.0, 'trials_per_subject': 200},   # 양 세션 = 200
}

for name, spec in REQUIRED.items():
    path = os.path.join(prep, f'{name}.npz')
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    d = np.load(path, allow_pickle=True)
    X        = d['X']
    y        = d['y']
    subjects = d['subjects']
    ch_names = d['ch_names']
    sfreq    = float(d['sfreq'])

    unique_subjs, counts = np.unique(subjects, return_counts=True)
    trials_set = sorted(set(counts.tolist()))

    print(f'\n{name}:')
    print(f'  X={X.shape}  subjects={len(unique_subjs)}  '
          f'ch={len(ch_names)}  sfreq={sfreq}')
    print(f'  trials/subject: {trials_set}')
    print(f'  labels: {np.unique(y, return_counts=True)}')

    # ── 검증 ──────────────────────────────────────────────────────────────────
    assert abs(sfreq - spec['sfreq']) < 1e-6, \
        f'{name}: sfreq={sfreq} ≠ {spec["sfreq"]}'

    assert X.shape[1] == spec['n_ch'], \
        f'{name}: channels={X.shape[1]} ≠ {spec["n_ch"]}'

    assert X.shape[2] == spec['n_times'], \
        f'{name}: n_times={X.shape[2]} ≠ {spec["n_times"]}'

    assert len(unique_subjs) == spec['n_subjects'], \
        f'{name}: {len(unique_subjs)} subjects ≠ {spec["n_subjects"]}'

    if spec['trials_per_subject'] is not None:
        wrong = [s for s, c in zip(unique_subjs, counts)
                 if c != spec['trials_per_subject']]
        if wrong:
            raise ValueError(
                f'{name}: subjects {wrong[:5]} have '
                f'{[int(counts[np.where(unique_subjs==s)[0][0]]) for s in wrong[:5]]} trials '
                f'— expected {spec["trials_per_subject"]} (both sessions). '
                f'\nFound trial counts: {trials_set}'
                f'\nThis NPZ appears to contain session 1 only. '
                f'Re-export with sessions=(1,2) or use preprocessed_raw_unified instead.'
            )

    assert len(y) == X.shape[0] and len(subjects) == X.shape[0], \
        f'{name}: length mismatch X={len(X)} y={len(y)} subjects={len(subjects)}'

    print(f'  ✅ {name} OK')

print('\n✅ All datasets validated — protocol matches legacy sfreq100 (200 trials/subject for Lee2019)')

MI_PREPROCESSED_DIR = /content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100
MI_BACKUP_DIR = /content/drive/MyDrive/MI_loso_project/colab_results/dsa_sea_snapshot_20260628
total 718M
-rw------- 1 root root 480M Jun 27 18:43 cho2017.npz
-rw------- 1 root root 239M Jun 27 18:12 lee2019.npz

cho2017: X=(10520, 64, 201), y=(10520,), subjects=52, channels=64, sfreq=100.0
labels: (array([0, 1]), array([5260, 5260]))
trials/subject: [200, 240] ...

lee2019: X=(5400, 62, 201), y=(5400,), subjects=54, channels=62, sfreq=100.0
labels: (array([0, 1]), array([2700, 2700]))
trials/subject: [100] ...


ValueError: lee2019 has X=(5400, 62, 201), expected (10800, 62, 201). Use the legacy sfreq100 preprocessed files, not a one-session MOABB export.

## 4. Dependency Sanity Check

Colab already includes the core stack. This cell only checks imports.

In [ ]:
import torch, scipy, sklearn
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
print('scipy', scipy.__version__)
print('sklearn', sklearn.__version__)
!python -m py_compile manage_colab_dsa_sea_snapshot_20260628.py cross_dataset.py

## 5. Run CSPNet Cho -> Lee

Set `FORCE=True` only when rerunning after a failed or interrupted attempt.

In [ ]:


!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models cspnet \
  --direction cho2lee \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 6. Save Current Results To Drive

In [ ]:
SAVE_DIR = os.environ['MI_BACKUP_DIR']
!mkdir -p "$SAVE_DIR"
!cp -v /content/MI_loso_project/project/crossdata/colab_dsa_sea_snapshot_20260628.md "$SAVE_DIR"/ 2>/dev/null || true
!cp -v /content/MI_loso_project/project/crossdata/results/loso_results_20260628_colab_dsa_sea_snapshot_*_cross_*.csv "$SAVE_DIR"/ 2>/dev/null || true
!ls -lh "$SAVE_DIR"

## 7. Run CSPNet Lee -> Cho

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models cspnet \
  --direction lee2cho \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 8. Optional: EEGNet Both Directions

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models eegnet \
  --direction both \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 9. Optional: Conformer Both Directions

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models conformer \
  --direction both \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 10. Logs And Summary

In [ ]:
%cd /content/MI_loso_project/project/crossdata
!cat colab_dsa_sea_snapshot_20260628.md || true
!ls -lh results/runs/*dsa_sea_snapshot*.log 2>/dev/null || true
!tail -n 80 results/runs/*dsa_sea_snapshot*.log 2>/dev/null || true
!ls -lh results/loso_results_20260628_colab_dsa_sea_snapshot_*_cross_*.csv 2>/dev/null || true

## 11. Push Verified Results To GitHub

Run this only after checking the backup folder contains the expected summary and CSV files. This step is intentionally manual and asks for a GitHub token at runtime.

In [ ]:
from getpass import getpass
import os

if not os.environ.get('GITHUB_TOKEN'):
    os.environ['GITHUB_TOKEN'] = getpass('GitHub token with contents:write permission: ')

%cd /content/MI_loso_project
!git pull --ff-only origin master
!mkdir -p project/crossdata/results
!cp -v "$MI_BACKUP_DIR"/colab_dsa_sea_snapshot_20260628.md project/crossdata/colab_dsa_sea_snapshot_20260628.md
!cp -v "$MI_BACKUP_DIR"/loso_results_20260628_colab_dsa_sea_snapshot_*.csv project/crossdata/results/
!git status --short
!git config user.name "heegyukim4043"
!git config user.email "55726335+heegyukim4043@users.noreply.github.com"
!git add project/crossdata/colab_dsa_sea_snapshot_20260628.md project/crossdata/results/loso_results_20260628_colab_dsa_sea_snapshot_*.csv
!git commit -m "Add Colab DSA SEA Snapshot results" || true
!git -c http.extraHeader="AUTHORIZATION: bearer $GITHUB_TOKEN" push origin master